# E6: EfficientNet-B1 with Adam Optimizer

1. Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np  
import os
import random

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

import torch
import torch.nn as nn   
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision import models
from PIL import Image

2. Device Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

3. Dataset Paths

In [ ]:
data_dir = "chest_xray"

train_dir = os.path.join(data_dir, "train") 
test_dir = os.path.join(data_dir, "test")
val_dir = os.path.join(data_dir, "val")

4. Transform

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),  
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

5. Load Dataset

In [ ]:
train_data = datasets.ImageFolder(train_dir, transform=train_transforms)
val_data = datasets.ImageFolder(val_dir, transform=val_test_transforms) 
test_data = datasets.ImageFolder(test_dir, transform=val_test_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = train_data.classes
print(f"Classes: {class_names}")

6. Show Random 6 images

In [ ]:
def show_sample_images(data_loader, class_names, num_images=6):
    images, labels = next(iter(data_loader))
    images = images[:num_images]
    labels = labels[:num_images]
    
    plt.figure(figsize=(12, 6))

    for i in range(num_images):
        plt.subplot(2, 3, i+1)
        img = images[i].permute(1, 2, 0).numpy()
        plt.imshow(img)
        plt.title(class_names[labels[i]])
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

show_sample_images(train_loader, class_names)

7. Load EfficientNet-B1

In [ ]:
model = models.efficientnet_b1(pretrained=True)

8. Freeze Layers

In [ ]:
for param in model.parameters():
    param.requires_grad = False

9. Modify Classifier

In [ ]:
model.add_module("classifier", nn.Linear(model.classifier[1].in_features, len(class_names)))
model = model.to(device)

10. Loss & Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

11. Training Loop + Metrics

In [ ]:
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

epochs = 50

best_val_acc = 0
patience = 3
patience_counter = 0
best_epoch = 0
best_model_state = None

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / total
    train_accuracy = correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_loss /= val_total
    val_accuracy = val_correct / val_total
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{epochs} - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} - "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}", end="")
    
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        best_epoch = epoch + 1
        patience_counter = 0
        best_model_state = model.state_dict().copy()
        print(" ✓ (Best)")
    else:
        patience_counter += 1
        print(f" (No improvement: {patience_counter}/{patience})")
        
        if patience_counter >= patience:
            print(f"\n🛑 Early Stopping at Epoch {epoch+1}")
            print(f"Best model found at Epoch {best_epoch} with Val Acc: {best_val_acc:.4f}")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✓ Best model loaded (Epoch {best_epoch})")

In [ ]:
print("\n" + "="*50)
print("EARLY STOPPING SUMMARY (E6 - EfficientNet-B1)")
print("="*50)
print(f"Best Epoch: {best_epoch}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Training Epochs Executed: {len(train_losses)}")
print("="*50)

12. Plot Loss

In [ ]:
plt.figure()
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('E6 (EfficientNet-B1) - Loss Curve')
plt.legend()
plt.show()

13. Plot Accuracy

In [ ]:
plt.figure()
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('E6 (EfficientNet-B1) - Accuracy Curve')
plt.legend()
plt.show()

14. Test Prediction

In [ ]:
y_true, y_pred = [], []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

In [ ]:
accuracy = np.mean(np.array(y_true) == np.array(y_pred))
print(f"E6 (EfficientNet-B1) - Test Accuracy: {accuracy:.4f}")

15. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
             xticklabels=class_names, 
             yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')  
plt.title('E6 (EfficientNet-B1) - Confusion Matrix')
plt.show()

16. Classification Report

In [ ]:
print("E6 (EfficientNet-B1) - Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

17. Save Model